# CSE 4095 — Class 2: Classification
Use AI as a coding agent, but inspect and explain every major decision.

## 1. Load the Class 1 dataset
Upload `class1_student_habits.csv`, import Pandas, load it as `df`, and inspect the first rows.

In [82]:
import pandas as pd

df = pd.read_csv("class1_student_habits.csv")

display(df.head())

classes = (df["exam_score"] >= 80).map({
    True: "Score ≥ 80",
    False: "Score < 80"
})

classes.value_counts().to_frame("Count")

,student_id,study_hours_per_week,sleep_hours_per_night,attendance_rate,practice_problems_per_week,prior_programming_experience,study_group,commute_minutes,quiz_average,exam_score
0,S001,1.3,7.7,71.8,8,Strong,No,0,67.0,64.6
1,S002,9.4,6.6,74.2,14,NaN,Yes,52,74.8,81.5
2,S003,11.2,6.1,70.3,15,Some,No,7,99.1,92.0
3,S004,10.8,7.6,77.1,16,Strong,No,12,86.4,100.0
4,S005,8.0,5.6,89.7,15,NaN,Yes,12,75.7,68.5


,Count
exam_score,
Score < 80,151
Score ≥ 80,99


## 2. Create the target
Create `high_exam_score`: Yes if `exam_score >= 80`, otherwise No. Then inspect class counts.

In [83]:
# TODO: create target and inspect class balance
# Create the categorical target
df["high_exam_score"] = (df["exam_score"] >= 80).map({
    True: "Yes",
    False: "No"
})

# Count students and calculate percentages for each class
class_counts = df["high_exam_score"].value_counts()

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": (class_counts / len(df) * 100).round(1)
})

display(class_summary)

,Count,Percentage
high_exam_score,,
No,151,60.4
Yes,99,39.6


1. Why is predicting exam_score a regression problem?
- It's because we're predicting a numerical value, regression problem predict quantities

2. Why is predicting high_exam_score a classification problem?
- Since we're classifying whether an exam score belongs to high_exam_score using Yes No logic, it is a binary classification problem

3. - 4. How many students belong to each class and what percentage?
- There's 250 students, and about 39.6% have high_exam_score while 60.4% do not.

## 3. Choose X and y
Use only legitimate features. Explain in Markdown why `student_id` and `exam_score` are excluded.

In [84]:
# TODO: define feature columns, X, and y
feature_columns = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes"
]

X = df[feature_columns].copy()
y = df["high_exam_score"].copy()

display(X.head())
display(y.head())

,study_hours_per_week,sleep_hours_per_night,attendance_rate,practice_problems_per_week,prior_programming_experience,study_group,commute_minutes
0,1.3,7.7,71.8,8,Strong,No,0
1,9.4,6.6,74.2,14,NaN,Yes,52
2,11.2,6.1,70.3,15,Some,No,7
3,10.8,7.6,77.1,16,Strong,No,12
4,8.0,5.6,89.7,15,NaN,Yes,12


0     No
1    Yes
2    Yes
3    Yes
4     No
Name: high_exam_score, dtype: str

1. Why is exam_score and student_id excluded from our list of features?
- student_id is excluded because on its own, it only identifies a student, nothing about a student's academic performance
- exam_score is excluded for a different reason, we want to determine if someone got a high_exam_score based on the features, if we give exam_score as a feature, a model will know the threshold for a high_exam_score, leading to target leakage.

## 4. Baseline
Calculate the majority-class baseline accuracy before training a model.

In [85]:
# TODO: calculate baseline
# Count each class in the target
class_counts = y.value_counts()

# Identify the most common class
majority_class = class_counts.idxmax()

# Accuracy from always predicting that class
baseline_accuracy = class_counts.max() / len(y)

print(f"Majority class: {majority_class}")
print(f"Baseline accuracy: {baseline_accuracy:.1%}")

Majority class: No
Baseline accuracy: 60.4%


1. What is a majority class?
- The majority class is the most common target category

2. What accuracy does the majority-class baseline achieve?
- It shows that always predicting No correctly classifies 151 student, and vice versa when Yes

3. Why is the baseline useful?
- A baseline gives us a simple benchmark assuming no student features, comparing our model against it shows whether we're learning from the features improve accuracy over just choosing the most common class.

## 5. Train/test split and preprocessing
Ask your AI agent for a pipeline that handles numerical/categorical missing values and one-hot encoding.

In [86]:
# TODO: imports, preprocessing, and train/test split
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Split the original data before fitting any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 2. Identify numerical and categorical features
numerical_features = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes"
]

categorical_features = [
    "prior_programming_experience",
    "study_group"
]

# 3. Numerical preprocessing:
# Replace missing values with the median, then standardize
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 4. Categorical preprocessing:
# Replace missing values with the most frequent category, then encode
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 5. Apply the appropriate pipeline to each group of columns
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print(f"Training students: {len(X_train)}")
print(f"Test students: {len(X_test)}")

Training students: 200
Test students: 50


1. What is the purpose of the training set?
- Our training set set serves to help the model learn relationships between features and high_exam score

2. What is the purpose of the test set?
- It serves to evaluation how well the trained model predicts outcomes given unseen data

3. What does stratification do?
- It keeps the proportions of the target class roughly the same in both categories, about 60% and 40%

4. Why shouldn't we train the model using the test data?
- We should avoid testing on test data because it serves as our evalulator and a honest way to check how well the model works with data it has never seen before

1. Why do categorical variables need to be converted before Logistic Regression we can use them?
- This is because Logisitc Regression uses mathematical calculations given numerical inputs, text categories must be represented as numbers

2. What does one-hot encoding do?
- One hot encoding creates a separate indicator column for each category, using 1 when the category applies and 0 otherwise

3. Why handle missing values?
- scikit-learn cannot use on missing values, which is why our pipeline replaces missing values with the training median and missing categorical values with the most frequent training category

4. What does StandardScaler do?
- It changes each numerical feature by subtracting its training mean and dividing by its training standard deviation

1. Why might scaling be useful for Logistic Regression?
- Some features like commute minutes and sleeop hours use different sclaes of time, by scaling, we can make these features comparable

2. Does standardization mean that all original values becomes numbers between 0 and 1?
- No, standscaler subtracts the training mean and divides by the training standard deviation. Values can be negative or greatern than 1. For example, a standardized value of -1 means on standard deviation below the training mean.

3. Why should the test set not be used to calculate the mean and standard deviation use for scaling?
- Like said before, we do this to avoid data leakage, which could allow test data to influence preprocessing

## 6. Logistic regression
Fit the model using training data only.

In [87]:
# TODO: build and fit pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Combine preprocessing and logistic regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Fit preprocessing and the classifier using training data only
model.fit(X_train, y_train)

print("Logistic regression trained successfully.")

Logistic regression trained successfully.


1. What does .fit() do?
- .fit() trains the pipeline using the data we provided earlier, using all of our preprocessing settings, transforms the training features, and trains logisitic regression to predict the training labels

2. What is learned from the training data?
- Medians for numerial features and most frequent categories for categorical features, also including numerical feature's mean and standard deviation, naming a few examples.

3. Why is it useful to preprocessing and the classifier into one pipeline?
- A pipeline on its own run the steps in the correct order and applies the same learned preprocessing when making predictions, it also helps prevent leakage, fitting on training data learns all settings there, while predicting on test data reuses those settings without recalculating them

## 7. Evaluate
Calculate accuracy, confusion matrix, precision, and recall. Compare accuracy with the baseline.

In [88]:
# TODO: evaluate test predictions
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

# Predict test labels using the trained pipeline
y_pred = model.predict(X_test)

# Calculate model metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, pos_label="Yes", zero_division=0
)
recall = recall_score(
    y_test, y_pred, pos_label="Yes", zero_division=0
)

# Choose the majority class from training data only
majority_class = y_train.value_counts().idxmax()

# Evaluate the baseline on the same test set
baseline_pred = pd.Series(majority_class, index=y_test.index)
test_baseline_accuracy = accuracy_score(y_test, baseline_pred)

results = pd.DataFrame({
    "Metric": [
        "Model accuracy",
        "Test baseline accuracy",
        "Precision (Yes)",
        "Recall (Yes)"
    ],
    "Value": [
        accuracy,
        test_baseline_accuracy,
        precision,
        recall
    ]
})

display(results.style.format({"Value": "{:.1%}"}))

# Rows = actual classes; columns = predicted classes
matrix = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])

matrix_df = pd.DataFrame(
    matrix,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"]
)

display(matrix_df)

difference = (accuracy - test_baseline_accuracy) * 100
print(f"Accuracy difference from baseline: {difference:+.1f} percentage points")

,Metric,Value
0,Model accuracy,84.0%
1,Test baseline accuracy,60.0%
2,Precision (Yes),83.3%
3,Recall (Yes),75.0%


,Predicted No,Predicted Yes
Actual No,27,3
Actual Yes,5,15


Accuracy difference from baseline: +24.0 percentage points


1. What is the difference between predict() and predict_proba()?
- predict() returns the predicted class given for each student, while predict_proba() returns the estimated probability of each class for each student

2. What does a predicted probability near 0.5 suggest?
- Our logistic regression model predicts with 50/50 probability that any exam_score is above or below 80

3. What does a probability of 0.95 detail?
- Our logistic regression model predicts with a 95% probability that a particular exam score is above 80 and 5% probability of it not

4. What fraction of all test predictions were correct?
- Only 42/50 of test predictions were correct

5. Among students predicted to have a high exam_score, what fraction actually did?
- Of the 18 predicted to have a high exam score, only 15/18 did have one

6. Among students who actually had a high exam score, what fraction did the model identify?
- Of the 20 who had high exam scores, only 15 were identified correctly or 15/20



## 8. Inspect individual predictions
Create a small table with actual class, predicted class, and x probability.

In [89]:
# TODO: inspect predictions
# Find the probability column corresponding to "Yes"
yes_index = list(model.classes_).index("Yes")

predictions = pd.DataFrame({
    "Actual class": y_test,
    "Predicted class": model.predict(X_test),
    "Probability of Yes": model.predict_proba(X_test)[:, yes_index]
})

# Display 10 predictions with row numbers from 0 to 9
display(
    predictions.head(10).reset_index(drop=True).style.format({
        "Probability of Yes": "{:.1%}"
    })
)

,Actual class,Predicted class,Probability of Yes
0,Yes,No,36.8%
1,No,No,10.7%
2,No,No,20.7%
3,No,No,29.1%
4,No,No,36.7%
5,Yes,Yes,64.6%
6,Yes,No,37.4%
7,No,No,21.3%
8,Yes,Yes,83.1%
9,No,No,28.0%


1. Do you notice anything interesting about the mistakes?
- When it came to predicting an person's exam score to be a high or not, the probability of saying yes to an actual class of "yes" was usually below our threshold of 0.5 but significantly higher than the our predictions that our classifier got the actual classifier correct, in this case when the actual class was "No". It showed that our classifier was a little bit unsure for this particular student, which ended up being wrong in terms of their prediction. (I'm referring to at index 0)

2. Can you form a hypothesis about why the model may have found some of these students difficult to classify?
- If a student's attendance_score was high, then the classifier could've been more likely to say yes to predicting their exam score being above 80 because previous performances can indicate how well they're likely to do on the exam like attendance.

## 9. Controlled leakage experiment
Temporarily include `exam_score` as a feature, compare results, and explain why this experiment is invalid as a real model.

In [90]:
# TODO: leakage experiment
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

# 1. Include exam_score deliberately to introduce target leakage
leakage_features = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
    "exam_score"
]

X_leakage = df[leakage_features].copy()
y_leakage = df["high_exam_score"].copy()

# 2. Repeat the original 80/20 split
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leakage,
    y_leakage,
    test_size=0.20,
    random_state=42,
    stratify=y_leakage
)

# Confirm that both experiments use the same test students
assert X_test_leak.index.equals(X_test.index)

# 3. Create fresh preprocessing pipelines
numerical_features_leak = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes",
    "exam_score"
]

categorical_features_leak = [
    "prior_programming_experience",
    "study_group"
]

numerical_pipeline_leak = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_leak = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_leak = ColumnTransformer([
    ("numerical", numerical_pipeline_leak, numerical_features_leak),
    ("categorical", categorical_pipeline_leak, categorical_features_leak)
])

# 4. Train a separate model using training data only
leakage_model = Pipeline([
    ("preprocessor", preprocessor_leak),
    ("classifier", LogisticRegression(max_iter=1000))
])

leakage_model.fit(X_train_leak, y_train_leak)

# 5. Predict with both models
original_pred = model.predict(X_test)
leakage_pred = leakage_model.predict(X_test_leak)

# Majority-class baseline, chosen from training data
majority_class_leak = y_train_leak.value_counts().idxmax()
baseline_pred_leak = pd.Series(
    majority_class_leak,
    index=y_test_leak.index
)

# 6. Compare results
def calculate_metrics(actual, predicted):
    return {
        "Accuracy": accuracy_score(actual, predicted),
        "Precision (Yes)": precision_score(
            actual, predicted, pos_label="Yes", zero_division=0
        ),
        "Recall (Yes)": recall_score(
            actual, predicted, pos_label="Yes", zero_division=0
        )
    }

comparison = pd.DataFrame({
    "Majority-class baseline": calculate_metrics(
        y_test_leak, baseline_pred_leak
    ),
    "Original model": calculate_metrics(y_test, original_pred),
    "Leakage model": calculate_metrics(y_test_leak, leakage_pred)
}).T

display(comparison.style.format("{:.1%}"))

# 7. Display the leakage model's confusion matrix
leakage_matrix = pd.DataFrame(
    confusion_matrix(
        y_test_leak, leakage_pred, labels=["No", "Yes"]
    ),
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"]
)

print("Leakage model confusion matrix:")
display(leakage_matrix)

difference = (
    comparison.loc["Leakage model", "Accuracy"]
    - comparison.loc["Original model", "Accuracy"]
) * 100

print(
    f"Leakage model accuracy difference from original: "
    f"{difference:+.1f} percentage points"
)

,Accuracy,Precision (Yes),Recall (Yes)
Majority-class baseline,60.0%,0.0%,0.0%
Original model,84.0%,83.3%,75.0%
Leakage model,98.0%,100.0%,95.0%


Leakage model confusion matrix:


,Predicted No,Predicted Yes
Actual No,30,0
Actual Yes,1,19


Leakage model accuracy difference from original: +14.0 percentage points


1. What happend to the accuracy?
- It shot up above the baseline by a significant amount

2. Why did this happen?
- This happened because we essentially told the model the actual exam_scores of each student, predicting high_exam_score became worthless because we can simply just compared the exam_score to the definition of what was considered a high_exam_score

3. Why is this model invalid even if its accuracy is extremely high?
-  It is because our model is no longer predicting high_exam_scores but rather relying on exam_score being present to effectively answer correctly whether a student has a high_exam_score or not

## 10. Your experiment
Change one legitimate modeling decision. Write your prediction before running it, then compare results.

In [91]:
# TODO: student-designed experiment
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

# 1. Load data and create the target
df = pd.read_csv("class1_student_habits.csv")

df["high_exam_score"] = (df["exam_score"] >= 80).map({
    True: "Yes",
    False: "No"
})

original_numerical = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes"
]

categorical = [
    "prior_programming_experience",
    "study_group"
]

original_features = original_numerical + categorical
quiz_features = original_features + ["quiz_average"]

X = df[quiz_features].copy()
y = df["high_exam_score"].copy()

# 2. Split once so both models use exactly the same students
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Build a fresh pipeline for each model
def build_model(numerical_columns):
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numerical", numerical_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

original_model = build_model(original_numerical)
quiz_model = build_model(original_numerical + ["quiz_average"])

# 4. Fit using training data only
original_model.fit(X_train[original_features], y_train)
quiz_model.fit(X_train, y_train)

# 5. Generate predictions
original_pred = original_model.predict(X_test[original_features])
quiz_pred = quiz_model.predict(X_test)

majority_class = y_train.value_counts().idxmax()
baseline_pred = pd.Series(majority_class, index=y_test.index)

# 6. Compare evaluation metrics
def calculate_metrics(actual, predicted):
    return {
        "Accuracy": accuracy_score(actual, predicted),
        "Precision (Yes)": precision_score(
            actual, predicted, pos_label="Yes", zero_division=0
        ),
        "Recall (Yes)": recall_score(
            actual, predicted, pos_label="Yes", zero_division=0
        )
    }

comparison = pd.DataFrame({
    "Majority-class baseline": calculate_metrics(y_test, baseline_pred),
    "Original model": calculate_metrics(y_test, original_pred),
    "Model with quiz_average": calculate_metrics(y_test, quiz_pred)
}).T

display(comparison.style.format("{:.1%}"))

# 7. Display confusion matrices for both models
for name, predicted in [
    ("Original model", original_pred),
    ("Model with quiz_average", quiz_pred)
]:
    matrix_df = pd.DataFrame(
        confusion_matrix(y_test, predicted, labels=["No", "Yes"]),
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"]
    )

    print(f"\n{name} — confusion matrix:")
    display(matrix_df)

# 8. Report the change in accuracy
original_accuracy = comparison.loc["Original model", "Accuracy"]
quiz_accuracy = comparison.loc["Model with quiz_average", "Accuracy"]
baseline_accuracy = comparison.loc["Majority-class baseline", "Accuracy"]

print(
    f"Change from original model: "
    f"{(quiz_accuracy - original_accuracy) * 100:+.1f} percentage points"
)
print(
    f"Difference from baseline: "
    f"{(quiz_accuracy - baseline_accuracy) * 100:+.1f} percentage points"
)

,Accuracy,Precision (Yes),Recall (Yes)
Majority-class baseline,60.0%,0.0%,0.0%
Original model,84.0%,83.3%,75.0%
Model with quiz_average,82.0%,78.9%,75.0%



Original model — confusion matrix:


,Predicted No,Predicted Yes
Actual No,27,3
Actual Yes,5,15



Model with quiz_average — confusion matrix:


,Predicted No,Predicted Yes
Actual No,26,4
Actual Yes,5,15


Change from original model: -2.0 percentage points
Difference from baseline: +22.0 percentage points


1. Did quiz_average improve the model?
- The accuracy and precision of the model with quiz_average decreased compared to the original model

2. Which metrics changed?
- Only accuracy and precision really changed

3. Was your original predictions correct?
- No, I thought it would improve the model, but it seemed to make the model slightly worse. 

My Additional Experiment, excluding quiz_average, commute_minutes, sleep_hours_per_night as feature inputs.

In [92]:
# Record your expected effect before running this experiment.

# 1. Keep five features; exclude quiz average, commute, and sleep
reduced_numerical = [
    "study_hours_per_week",
    "attendance_rate",
    "practice_problems_per_week"
]

reduced_features = reduced_numerical + categorical

# 2. Create a fresh pipeline and train using training data only
reduced_model = build_model(reduced_numerical)

reduced_model.fit(
    X_train[reduced_features],
    y_train
)

# 3. Predict on the same test students
reduced_pred = reduced_model.predict(X_test[reduced_features])

# 4. Compare all models using the earlier metrics function
experiment_comparison = pd.DataFrame({
    "Majority-class baseline": calculate_metrics(y_test, baseline_pred),
    "Original model": calculate_metrics(y_test, original_pred),
    "Model with quiz_average": calculate_metrics(y_test, quiz_pred),
    "Reduced model": calculate_metrics(y_test, reduced_pred)
}).T

display(experiment_comparison.style.format("{:.1%}"))

# 5. Display the reduced model's confusion matrix
reduced_matrix = pd.DataFrame(
    confusion_matrix(y_test, reduced_pred, labels=["No", "Yes"]),
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"]
)

print("Reduced model — confusion matrix:")
display(reduced_matrix)

# 6. Report accuracy differences
reduced_accuracy = experiment_comparison.loc["Reduced model", "Accuracy"]

for reference in [
    "Majority-class baseline",
    "Original model",
    "Model with quiz_average"
]:
    reference_accuracy = experiment_comparison.loc[reference, "Accuracy"]
    difference = (reduced_accuracy - reference_accuracy) * 100

    print(
        f"Reduced model vs {reference}: "
        f"{difference:+.1f} percentage points"
    )

,Accuracy,Precision (Yes),Recall (Yes)
Majority-class baseline,60.0%,0.0%,0.0%
Original model,84.0%,83.3%,75.0%
Model with quiz_average,82.0%,78.9%,75.0%
Reduced model,80.0%,77.8%,70.0%


Reduced model — confusion matrix:


,Predicted No,Predicted Yes
Actual No,26,4
Actual Yes,6,14


Reduced model vs Majority-class baseline: +20.0 percentage points
Reduced model vs Original model: -4.0 percentage points
Reduced model vs Model with quiz_average: -2.0 percentage points


Before changing this code, I think this reduced model will perform better than the previous 2 experiment mainly because we are removing possibly "weak" and "unrelated" input features that could contribute to the predictor's performance

It seems actually this model performed the worst in terms of Accuracy and Precision, but again, not by a significant amount. This is probably because now the model has less features to use as an indicator to predict high_exam_score.

## Conclusion
Write a short evidence-based conclusion and AI Reflection here.


The AI helped me figure out how the interpret confusion matrices, which was helpful because I don't work with machine learning models much, so it was helpful asking the AI to help me determine which of the results did the model guess correctly given actual No's and Yes's.

1. What did the AI suggest?
- The AI simply just showed me the mathematical computation and showed me the probabilities given the confusion matrix 

2. Did you accept the suggestion?
- I did, and I used its answer to help me respond to the questions being asked of me

3. How did you verify the suggestion was reasonable?
- I performed the computation, and double checked by searching on the internet how to interpret confusion matrices

4. Did the AI make any assumption that you needed to question or correct?
- Not particularly, since I gave a lot of my data to the AI and it could run the programs on its own to check

5. What did you learn?
- AI is powerful tool in ML and it made conducting experiments of my own pretty easy given that there are guidelines present